In [1]:

# Cell 1

import numpy as np
import pandas as pd
import torch
import joblib
import torch

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [2]:
import pandas as pd

data_path = "/kaggle/input/datasets/joelleiliovits/new-data-csv/new_data.csv"
df = pd.read_csv(data_path)

In [3]:
df = df[["text", "tags"]].copy()
df = df.dropna(subset=["text", "tags"])

df["text"] = df["text"].astype(str).str.strip()
df["tags"] = df["tags"].astype(str).str.strip()

df = df[(df["text"] != "") & (df["tags"] != "")]



In [4]:
model_name = "distilbert-base-uncased"
model_name = "FacebookAI/roberta-base"


In [5]:
df["label_list"] = df["tags"].apply(lambda x: x.split())
df[["text", "tags", "label_list"]].head()  


,text,tags,label_list
0,loop through elements of list in a pandas data...,python pandas,"[python, pandas]"
1,ssl error when running pip search in python 2....,python ssl,"[python, ssl]"
2,how do i clear errno in c# how do i clear errn...,c# linux,"[c#, linux]"
3,segmentation fault as soon the binary launch h...,linux debugging,"[linux, debugging]"
4,changing data in jsp using ajax i have jsp pag...,javascript java,"[javascript, java]"


In [6]:
df["num_labels"] = df["label_list"].apply(len)

print(df["num_labels"].value_counts().sort_index())
df[["text", "tags", "label_list", "num_labels"]].head()

num_labels
1    1000000
2     701000
3     112707
4       7949
5        304
Name: count, dtype: int64


,text,tags,label_list,num_labels
0,loop through elements of list in a pandas data...,python pandas,"[python, pandas]",2
1,ssl error when running pip search in python 2....,python ssl,"[python, ssl]",2
2,how do i clear errno in c# how do i clear errn...,c# linux,"[c#, linux]",2
3,segmentation fault as soon the binary launch h...,linux debugging,"[linux, debugging]",2
4,changing data in jsp using ajax i have jsp pag...,javascript java,"[javascript, java]",2


In [7]:
df_1 = df[df["num_labels"] == 1].copy()
df_2 = df[df["num_labels"] == 2].copy()
df_3_plus = df[df["num_labels"] >= 3].copy()



In [8]:
RANDOM_STATE = 42

df_1_sample = df_1.sample(
    n=min(850000, len(df_1)),
    random_state=RANDOM_STATE
).copy()

df_2_sample = df_2.sample(
    n=min(320000, len(df_2)),
    random_state=RANDOM_STATE
).copy()

df_final = pd.concat([df_1_sample, df_2_sample, df_3_plus], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Final shape:", df_final.shape)
print(df_final["num_labels"].value_counts().sort_index())

Final shape: (1290960, 4)
num_labels
1    850000
2    320000
3    112707
4      7949
5       304
Name: count, dtype: int64


In [9]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_final["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 52
y shape: (1290960, 52)
First labels: ['apache' 'asp.net-core' 'authentication' 'azure' 'bash' 'c#'
 'computer-vision' 'cors' 'cuda' 'debugging' 'deep-learning' 'django'
 'dns' 'docker' 'fastapi' 'firewall' 'gpu' 'http' 'inference' 'java']


In [10]:
X = df_final["text"].tolist()



In [11]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True
)



In [12]:
from datasets import Dataset
import numpy as np

y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

train_dataset = Dataset.from_dict({
    "text": X_train if isinstance(X_train, list) else X_train.tolist(),
    "labels": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_val if isinstance(X_val, list) else X_val.tolist(),
    "labels": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test if isinstance(X_test, list) else X_test.tolist(),
    "labels": y_test.tolist()
})

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1032768
Val: 129096
Test: 129096


In [13]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
# df فيه عمود text = title + body
sample_df = df.sample(n=min(100_000, len(df)), random_state=42).copy()

texts = sample_df["text"].fillna("").astype(str).tolist()

lengths = []
batch_size = 512

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    enc = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False
    )
    lengths.extend(len(x) for x in enc["input_ids"])

lengths = np.array(lengths)

print("count:", len(lengths))
print("min:", lengths.min())
print("median:", int(np.percentile(lengths, 50)))
print("p90:", int(np.percentile(lengths, 90)))
print("p95:", int(np.percentile(lengths, 95)))
print("p99:", int(np.percentile(lengths, 99)))
print("max:", lengths.max())

print("over_64 :", round((lengths > 64).mean() * 100, 2), "%")
print("over_128:", round((lengths > 128).mean() * 100, 2), "%")
print("over_256:", round((lengths > 256).mean() * 100, 2), "%")

Token indices sequence length is longer than the specified maximum sequence length for this model (735 > 512). Running this sequence through the model will result in indexing errors


count: 100000
min: 13
median: 203
p90: 502
p95: 625
p99: 885
max: 1942
over_64 : 95.36 %
over_128: 74.24 %
over_256: 37.1 %


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=150
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1032768 [00:00<?, ? examples/s]

Map:   0%|          | 0/129096 [00:00<?, ? examples/s]

Map:   0%|          | 0/129096 [00:00<?, ? examples/s]

In [16]:
num_labels = y_train.shape[1]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print("Num labels:", num_labels)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Num labels: 52


In [17]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/unixcode_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",    
    save_total_limit=3,
    save_only_model=False,

    per_device_train_batch_size=120,
    per_device_eval_batch_size=120,

    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1_at_3",
    greater_is_better=True,

    fp16=True,
    bf16=False,

    report_to="none"
)

In [18]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

THRESHOLD = 0.35

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    results = {}

    probs = sigmoid(logits)
    preds_thr = (probs >= THRESHOLD).astype(int)

    results["precision_threshold"] = precision_score(labels, preds_thr, average="micro", zero_division=0)
    results["recall_threshold"] = recall_score(labels, preds_thr, average="micro", zero_division=0)
    results["f1_threshold"] = f1_score(labels, preds_thr, average="micro", zero_division=0)

    for k in [1, 2, 3, 4, 5]:
        preds_k = top_k_binary_predictions(logits, k)

        results[f"precision_at_{k}"] = precision_score(labels, preds_k, average="micro", zero_division=0)
        results[f"recall_at_{k}"] = recall_score(labels, preds_k, average="micro", zero_division=0)
        results[f"f1_at_{k}"] = f1_score(labels, preds_k, average="micro", zero_division=0)

    return results

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [20]:
from pathlib import Path
import shutil
import re
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = Path("/kaggle/working/deberta_pair_cls")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_step(path):
    match = re.search(r"checkpoint-(\d+)", path.name)
    return int(match.group(1)) if match else -1

input_checkpoints = [
    p for p in Path("/kaggle/input").rglob("checkpoint-*")
    if p.is_dir()
]

input_checkpoints = sorted(input_checkpoints, key=checkpoint_step)

if len(input_checkpoints) == 0:
    print("No checkpoints found in /kaggle/input.")
    print("Training will start from scratch.")
else:
    print(f"Found {len(input_checkpoints)} checkpoint(s) in /kaggle/input:")

    for p in input_checkpoints:
        print(" -", p)

    print("\nCopying checkpoints to:", OUTPUT_DIR)

    for ckpt in input_checkpoints:
        dst = OUTPUT_DIR / ckpt.name

        if dst.exists():
            print(f"Already exists, skipping: {dst}")
            continue

        shutil.copytree(ckpt, dst)
        print(f"Copied: {ckpt.name}")

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
if last_checkpoint is None:
    print("\nNo checkpoint available in OUTPUT_DIR.")
    print("Next training cell should start from scratch.")
else:
    print("\nLast checkpoint available:")
    print(last_checkpoint)
    print("Next training cell should resume from this checkpoint.")

Found 3 checkpoint(s) in /kaggle/input:
 - /kaggle/input/notebooks/mikeelio4/notebook158576182f/unixcode_results/checkpoint-8608
 - /kaggle/input/notebooks/mikeelio4/notebook158576182f/unixcode_results/checkpoint-12912
 - /kaggle/input/notebooks/mikeelio4/notebook158576182f/unixcode_results/checkpoint-17216

Copying checkpoints to: /kaggle/working/deberta_pair_cls
Copied: checkpoint-8608
Copied: checkpoint-12912
Copied: checkpoint-17216

Last checkpoint available:
/kaggle/working/deberta_pair_cls/checkpoint-17216
Next training cell should resume from this checkpoint.


In [ ]:
if last_checkpoint is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=last_checkpoint)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision Threshold,Recall Threshold,F1 Threshold,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
1,0.089639,0.046880,0.829857,0.865735,0.847416,0.920772,0.639904,0.755065,0.624663,0.868238,0.726580,0.449988,0.938178,0.608240,0.346254,0.962537,0.509298,0.279961,0.972814,0.434794
2,0.046134,0.043647,0.830058,0.879801,0.854206,0.926357,0.643786,0.759645,0.628970,0.874224,0.731590,0.453296,0.945074,0.612711,0.348328,0.968303,0.512349,0.281536,0.978289,0.437241
3,0.042675,0.042503,0.825146,0.889044,0.855904,0.930385,0.646585,0.762948,0.630883,0.876883,0.733815,0.454437,0.947453,0.614253,0.349074,0.970376,0.513445,0.282043,0.980049,0.438028
4,0.040645,0.042024,0.830246,0.887424,0.857883,0.930749,0.646838,0.763247,0.631340,0.877519,0.734347,0.454814,0.948239,0.614763,0.349240,0.970839,0.513690,0.282188,0.980555,0.438254


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [21]:
if last_checkpoint is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=last_checkpoint)

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision Threshold,Recall Threshold,F1 Threshold,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
5,0.039292,0.041674,0.831461,0.888092,0.858844,0.931663,0.647473,0.763996,0.631952,0.878369,0.735059,0.455036,0.948702,0.615063,0.349432,0.971372,0.513972,0.282300,0.980943,0.438427


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


In [22]:


val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation Results: {'eval_loss': 0.04167382791638374, 'eval_precision_threshold': 0.8314609573057945, 'eval_recall_threshold': 0.8880915594937526, 'eval_f1_threshold': 0.8588437410521383, 'eval_precision_at_1': 0.9316632583503749, 'eval_recall_at_1': 0.6474733391114293, 'eval_f1_at_1': 0.7639961252004891, 'eval_precision_at_2': 0.6319521906178348, 'eval_recall_at_2': 0.8783692849337045, 'eval_f1_at_2': 0.735058598809328, 'eval_precision_at_3': 0.45503604552684307, 'eval_recall_at_3': 0.9487023508955151, 'eval_f1_at_3': 0.6150629878526543, 'eval_precision_at_4': 0.3494318181818182, 'eval_recall_at_4': 0.9713715082445534, 'eval_f1_at_4': 0.5139722250310834, 'eval_precision_at_5': 0.2822999938030613, 'eval_recall_at_5': 0.9809430498656861, 'eval_f1_at_5': 0.43842745588212295, 'eval_runtime': 372.2406, 'eval_samples_per_second': 346.808, 'eval_steps_per_second': 1.445, 'epoch': 5.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test Results: {'eval_loss': 0.04160656780004501, 'eval_precision_threshold': 0.8315094387176449, 'eval_recall_threshold': 0.8868143624370339, 'eval_f1_threshold': 0.8582718952752095, 'eval_precision_at_1': 0.9313456652413707, 'eval_recall_at_1': 0.6463688021804928, 'eval_f1_at_1': 0.7631200632162204, 'eval_precision_at_2': 0.6324866765817686, 'eval_recall_at_2': 0.8779117588555638, 'eval_f1_at_2': 0.7352596211208788, 'eval_precision_at_3': 0.4555421288550123, 'eval_recall_at_3': 0.9484605914640375, 'eval_f1_at_3': 0.6154742447684549, 'eval_precision_at_4': 0.35019868934746234, 'eval_recall_at_4': 0.9721739878395596, 'eval_f1_at_4': 0.5149139304410468, 'eval_precision_at_5': 0.28288870298072755, 'eval_recall_at_5': 0.9816464440657373, 'eval_f1_at_5': 0.43920754594446376, 'eval_runtime': 371.7938, 'eval_samples_per_second': 347.225, 'eval_steps_per_second': 1.447, 'epoch': 5.0}
